In [1]:
# ================================================================
# STEP 1. 라이브러리 설치 & 임포트
# ================================================================
# sentence-transformers : VectorDB 임베딩 + Reranker 모델 둘 다 포함
# chromadb              : VectorDB

import chromadb
from chromadb.utils import embedding_functions

# CrossEncoder : Reranker 모델 클래스
# 질문 + 문서를 함께 넣어서 관련도 점수를 계산
from sentence_transformers import CrossEncoder

In [2]:
# ================================================================
# STEP 2. VectorDB 구축 (상품 데이터 저장)
# ================================================================
# ----------------------------------------------------------------
# 상품 데이터 준비
# ----------------------------------------------------------------
# 노트북 관련 상품을 일부러 비슷하게 만들어둠
# → VectorDB 단독으로는 구분이 어려운 케이스를 만들기 위해
# → Reranker 가 얼마나 정확하게 의도를 파악하는지 확인하기 위해
products = [
    {
        "id": "p1",
        "name": "프로그래밍용 고성능 노트북",
        "desc": "최신 14세대 CPU와 32GB RAM을 탑재하여 무거운 딥러닝 및 개발 작업에 최적화된 랩탑입니다."
    },
    {
        "id": "p2",
        "name": "초경량 사무용 랩탑",
        "desc": "1kg 미만의 가벼운 무게로 카페나 도서관에서 문서 작업 및 웹서핑을 하기에 매우 좋은 노트북."
    },
    {
        "id": "p3",
        "name": "노트북 전용 가방",
        "desc": "15인치 노트북과 충전기, 마우스를 수납할 수 있는 방수 소재의 노트북 가방입니다."
    },
    {
        "id": "p4",
        "name": "노트북 거치대",
        "desc": "노트북을 눈높이에 맞게 올려주는 알루미늄 거치대. 목 디스크 예방과 자세 교정에 도움."
    },
    {
        "id": "p5",
        "name": "게이밍 노트북",
        "desc": "RTX 4080 그래픽카드와 240Hz 디스플레이를 탑재한 고사양 게이밍 랩탑."
    },
    {
        "id": "p6",
        "name": "방수 트레킹화",
        "desc": "고어텍스 소재로 비오는 날이나 거친 산악 지형에서도 발을 쾌적하게 보호하는 등산용 신발."
    },
    {
        "id": "p7",
        "name": "전문가용 미러리스 카메라",
        "desc": "4K 60fps 동영상 촬영과 빠르고 정확한 AF를 지원하여 유튜버 및 프로 사진작가에게 적합한 카메라."
    },
    {
        "id": "p8",
        "name": "노트북 쿨러 패드",
        "desc": "노트북 발열을 잡아주는 듀얼 팬 쿨러 패드. 장시간 작업 시 성능 저하를 방지합니다."
    },
]

In [3]:
# ----------------------------------------------------------------
# ChromaDB 설정 & 데이터 저장
# ----------------------------------------------------------------
# 인메모리 클라이언트 생성
client = chromadb.Client()

# 한국어 임베딩 모델 설정
st_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)

# 기존 컬렉션 있으면 삭제
try:
    client.delete_collection('reranking_demo')
except Exception:
    pass

# 컬렉션 저장
collection = client.create_collection(
    name = 'reranking_demo',
    embedding_function = st_ef,
    metadata={'hnsw:space': 'cosine'}
)

# 데이터 저장
collection.add(
    ids = [p['id'] for p in products],
    documents = [p['desc'] for p in products],
    metadatas = [{'name': p['name']} for p in products]
)

print(f'상품 {collection.count()}개 VectorDB 저장 완료')
for p in products:
    print(f'  {p["id"]} | {p["name"]}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

상품 8개 VectorDB 저장 완료
  p1 | 프로그래밍용 고성능 노트북
  p2 | 초경량 사무용 랩탑
  p3 | 노트북 전용 가방
  p4 | 노트북 거치대
  p5 | 게이밍 노트북
  p6 | 방수 트레킹화
  p7 | 전문가용 미러리스 카메라
  p8 | 노트북 쿨러 패드


In [6]:
# ================================================================
# STEP 3. VectorDB 1차 검색
# ================================================================

query = "개발 작업하기 좋은 노트북 추천해줘"

results = collection.query(
    query_texts=[query],
    n_results=8 # 전체 상품 다 가져오기
)
print(f'검색어: "{query}"')
print()
print('[ VectorDB 1차 검색 결과 ]')
print('-' * 60)

candidates = []

for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), start=1):
# enumerate(..., start=1) : 인덱스 번호를 1부터 세겠다
    similarity =1 - dist

    # 후보 리스트에 저장
    candidates.append({
        'name': meta['name'],
        'desc': doc,
        'similarity': similarity
    })

    print(f'{i}위 | 유사도: {similarity:.4f} | {meta["name"]}')

검색어: "개발 작업하기 좋은 노트북 추천해줘"

[ VectorDB 1차 검색 결과 ]
------------------------------------------------------------
1위 | 유사도: 0.3755 | 노트북 전용 가방
2위 | 유사도: 0.3240 | 노트북 쿨러 패드
3위 | 유사도: 0.3216 | 노트북 거치대
4위 | 유사도: 0.2863 | 초경량 사무용 랩탑
5위 | 유사도: 0.2073 | 게이밍 노트북
6위 | 유사도: 0.1982 | 프로그래밍용 고성능 노트북
7위 | 유사도: 0.1472 | 방수 트레킹화
8위 | 유사도: 0.1168 | 전문가용 미러리스 카메라


In [9]:
# ================================================================
# STEP 4. Reranker 2차 정제
# ================================================================

# ----------------------------------------------------------------
# Reranker 모델 로드
# ----------------------------------------------------------------

# CrossEncoder : 질문 + 문서를 함께 넣어서 관련도 점수 계산
# 'cross-encoder/ms-marco-MiniLM-L-6-v2'
#   - 영어 기반이지만 한국어도 어느 정도 작동
#   - 가볍고 빠른 경량 모델
#   - 처음 실행 시 모델 자동 다운로드 (수백 MB)
# 'bongsoo/kpf-cross-encoder-v1' 한국어 모델
reranker = CrossEncoder('bongsoo/kpf-cross-encoder-v1')

# ----------------------------------------------------------------
# Reranker 에 질문 + 후보 문서 쌍으로 입력
# ----------------------------------------------------------------

# CrossEncoder 입력 형식:
#   [(질문, 문서1), (질문, 문서2), ...]
#   → 둘을 함께 보기 때문에 문맥 파악이 더 정확함
pairs = [
    (query, candidate['desc'])
    for candidate in candidates
]

print(f'\nReranker 입력 쌍 ({len(pairs)}개):')
for i, (q, doc) in enumerate(pairs, start=1):
    print(f'  {i}. ("{q[:10]}...", "{doc[:20]}...")')

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\p3.10\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--bongsoo--kpf-cross-encoder-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/456M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/377 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/456M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


Reranker 입력 쌍 (8개):
  1. ("개발 작업하기 좋은...", "15인치 노트북과 충전기, 마우스를 ...")
  2. ("개발 작업하기 좋은...", "노트북 발열을 잡아주는 듀얼 팬 쿨러...")
  3. ("개발 작업하기 좋은...", "노트북을 눈높이에 맞게 올려주는 알루...")
  4. ("개발 작업하기 좋은...", "1kg 미만의 가벼운 무게로 카페나 ...")
  5. ("개발 작업하기 좋은...", "RTX 4080 그래픽카드와 240H...")
  6. ("개발 작업하기 좋은...", "최신 14세대 CPU와 32GB RA...")
  7. ("개발 작업하기 좋은...", "고어텍스 소재로 비오는 날이나 거친 ...")
  8. ("개발 작업하기 좋은...", "4K 60fps 동영상 촬영과 빠르고...")


In [10]:
# ----------------------------------------------------------------
# Reranker 점수 계산
# ----------------------------------------------------------------

# reranker.predict() : 각 (질문, 문서) 쌍의 관련도 점수 계산
# 반환값: 각 쌍에 대한 점수 리스트
#   - 점수가 높을수록 질문과 문서가 관련있음
#   - 점수 범위는 모델마다 다름 (보통 -10 ~ 10 사이)
rerank_scores = reranker.predict(pairs)

# ----------------------------------------------------------------
# Reranker 점수 기준으로 재정렬
# ----------------------------------------------------------------

# candidates 리스트에 rerank 점수를 추가
for i, score in enumerate(rerank_scores):
    candidates[i]['rerank_score'] = float(score)

# rerank_score 기준으로 내림차순 정렬
reranked = sorted(candidates, key=lambda x : -x['rerank_score'])

print('\n[ Reranker 2차 정제 결과 ]')
print('-' * 60)
for i, item in enumerate(reranked, start=1):
    print(f'{i}위 | Rerank점수: {item["rerank_score"]:6.2f} | {item["name"]}')


[ Reranker 2차 정제 결과 ]
------------------------------------------------------------
1위 | Rerank점수:   0.42 | 초경량 사무용 랩탑
2위 | Rerank점수:   0.36 | 프로그래밍용 고성능 노트북
3위 | Rerank점수:   0.25 | 게이밍 노트북
4위 | Rerank점수:   0.19 | 노트북 전용 가방
5위 | Rerank점수:   0.16 | 노트북 쿨러 패드
6위 | Rerank점수:   0.15 | 노트북 거치대
7위 | Rerank점수:   0.01 | 전문가용 미러리스 카메라
8위 | Rerank점수:   0.01 | 방수 트레킹화


Advanced RAG 흐름:

  ① VectorDB 1차 검색  →  후보 넉넉하게 추출 (빠름)
            ↓
  ② Reranker 2차 정제  →  진짜 의도에 맞는 것만 선별 (정확)
            ↓
  ③ LLM 에게 전달      →  정제된 문서 기반으로 답변 생성

VectorDB 단독 문제:
  "노트북" 단어만 보고 가방/거치대/쿨러도 상위에 올림

Reranker 적용 후:
  "개발 작업" 이라는 진짜 의도를 파악해서
  실제 노트북 제품만 상위에 올림

실무 포인트:
  - VectorDB 는 빠른 대신 정확도가 떨어질 수 있음
  - Reranker 는 느린 대신 의도 파악이 정확함
  - 둘을 조합하면 속도 + 정확도 동시에 확보
  - 한국어 서비스라면 반드시 한국어 모델 사용
''')